### LunarLander

飞船从空中下降，智能体要通过主发动机和左右姿态喷口，让它尽量平稳地落在两个旗帜之间。它比 CartPole 更接近真实控制问题，因为智能体不仅要“活得久”，还要同时控制位置、速度、角度、燃料消耗和落地方式。

#### 1. DQN 完整实现

[Code](./dqn_cartpole.py)

DQN 四大组件：

|组件|位置|作用|
|---|---|---|
|`QNetwork`|:41-63|MLP：`state→128→128→action`，输出每个动作的 Q 值|
|`ReplayBuffer`|:69-106|经验回放，打破数据相关性|
|`DQNAgent`|:112-216|核心智能体，整合 Q 网络 + 目标网络 + ε-greedy|
|`train()`|:222-358|训练循环 + 可视化 + 测试|

DQN 更新公式（`update` 方法）

In [ ]:
# 当前 Q 值：q_net 预测
q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

# 目标值：target_net 计算 r + γ·max_a' Q_target(s', a')
with torch.no_grad():
    next_q_max = self.target_net(next_states).max(dim=1)[0]
    targets = rewards + self.gamma * next_q_max * (1 - dones)

loss = nn.MSELoss()(q_values, targets)

三个稳定训练的技巧：
- 目标网络（:209-211）：延迟更新，避免"追自己尾巴"
- 经验回放（:84-103）：随机采样打破时序相关
- 梯度裁剪（:204）：`clip_grad_norm_(max_norm=10)` 防止梯度爆炸

#### 2. Double DQN 对比

[Code](./double_dqn_cartpole.py)

与标准 DQN 的唯一区别在于目标值计算（DoubleDQNAgent.update，:182-226）：

In [ ]:
# 标准 DQN：选动作 + 评价值用同一个网络 → 过估计
next_q_max = self.target_net(next_states).max(dim=1)[0]

# Double DQN：解耦"选动作"与"评价值"
best_actions = self.q_net(next_states).argmax(dim=1, keepdim=True)      # q_net 选动作
next_q_values = self.target_net(next_states).gather(1, best_actions)    # target_net 评价值
targets = rewards + self.gamma * next_q_values * (1 - dones)

#### 3. Gymnasium DQN 训练 （★ 最常用

[Code](./dqn_gym_sb3.py)


In [ ]:
# 1. 创建环境
env = make_vec_env(args.env_id, n_envs=1, seed=args.seed)
eval_env = Monitor(gym.make(args.env_id))

# 2. 构建 DQN 模型
model = DQN("MlpPolicy", env, 
    learning_rate=1e-3, buffer_size=100_000,
    batch_size=64, gamma=0.99,
    exploration_fraction=0.2, exploration_final_eps=0.05,
    tensorboard_log=str(run_dir / "tb"), ...)

# 3. 回调链：Checkpoint + EvalCSV + SwanLab
callbacks = [checkpoint_callback, eval_callback, swanlab_callback]

# 4. 训练 + 保存
model.learn(total_timesteps=..., callback=CallbackList(callbacks))
model.save(run_dir / "final_model")

自定义回调：
- EvaluationCsvCallback（:67-90）：每次评估写入 CSV，供后续画曲线
- SwanLabEvalCallback（:93-129）：将 CSV 中最新行同步到 SwanLab
- RestoreStdoutLog（:132-147）：修复 SwanLab 覆盖 stdout 日志的问题

#### 4. Atari DQN

[Code](./dqn_atari_sb3.py)

与上一个的关键差异在 环境构建（build_env，:229-259）：

In [ ]:
env = make_atari_env(env_id, wrapper_kwargs={...})  # Atari 包装器
env = VecFrameStack(env, n_stack=4)                 # 4 帧堆叠
return VecTransposeImage(env)                       # 转置为 CnnPolicy 格式

模型用 `CnnPolicy`（卷积网络）而非 `MlpPolicy`，学习率降为 `1e-4`，增加 `optimize_memory_usage` 选项。

#### 5. 自定义 Gymnasium 环境

[Code](./dqn_pokemon_red_pyboy.py)

核心是 `PokemonRedDQNEnv` 类（:105-284），展示了如何为任意游戏封装 Gymnasium 环境：

In [ ]:
class PokemonRedDQNEnv(gym.Env):
    # 观测：84×84 灰度屏幕图像
    observation_space = spaces.Box(shape=(84, 84, 1), dtype=np.uint8)
    # 动作：noop/up/down/left/right/a/b/start
    action_space = spaces.Discrete(8)
    
    def step(self, action):
        # 读取游戏内存坐标、徽章数、队伍数
        position = self._position()  # 读取 0xD361/0xD362/0xD35E
        reward = -0.002              # 每步微惩罚
        if position not in self.visited: reward += 0.08   # 探索新位置
        if map_changed: reward += 1.0                       # 到达新地图
        if badges_changed: reward += 5.0                    # 获得新徽章
        ...

#### 渲染与导出工具

|文件|功能|核心函数|
|---|---|---|
|`render_lunarlander.py`|LunarLander → GIF|`render_episode()` :11-32，循环 `env.render()` + `model.predict()`|
|`render_lunarlander_split.py`|每回合单独 GIF|加 `downsample_frames()` 控制帧数 :37-41|
|`render_atari.py`|Atari → GIF|`make_render_env()` :34-52，镜像训练时的包装器|
|`export_dqn_curves.py`|CSV → PNG 曲线|`export_curve()` :50-65，读 CSV 画带误差带的曲线|